# EECE 5554 Lab 4: Dead Reckoning with an IMU, GPS, and LiDAR in the Mustang Mach-E

## Learning objectives

Lab 4 takes your two drivers on the road. You will start your Lab 2 RTK GNSS driver and your Lab 3 VN-100 driver from one launch file, rehearse the whole data collection on foot, and then record the course's Ford Mustang Mach-E driving through Seattle, while the car records its own LiDAR, IMU, and wheel speeds. From those recordings you will estimate the car's heading, speed, and path from the IMU alone, then from the LiDAR, and measure how long each estimate stays close to the RTK track, which is good to a few centimeters.

By the end of this lab a well-prepared student will be able to:
<ul>
    <li> Start two sensor drivers from one ROS 2 launch file, and record both into one rosbag, while a second computer records the car </li>
    <li> Find and fix data collection problems on foot, before they cost a drive </li>
    <li> Calibrate a magnetometer for hard-iron and soft-iron distortion with a least-squares fit you write yourself </li>
    <li> Estimate heading from a magnetometer, a gyroscope, and a complementary filter that combines them, and explain where each one fails </li>
    <li> Estimate forward velocity from an accelerometer, correct it for bias, stops, and hills, and compare it with GPS </li>
    <li> Test a vehicle motion model against your data, and use it to find where the IMU sits in the car </li>
    <li> Dead-reckon a car's path and measure how long it stays close to an RTK reference </li>
    <li> Judge how good a GNSS reference is from its fix quality, and account for where the antenna sits on the car </li>
    <li> Line up the clocks of two computers from the data alone, by matching what two gyros saw </li>
    <li> Track the car with LiDAR scan matching, using an ICP you write yourself, and find where it fails </li>
    <li> Compare the IMU, the LiDAR, and the two together during a GNSS outage, and explain why real cars combine them </li>
</ul>

## Prerequisites

| Item | Explain |
|---|---|
| Lab 2 | Your RTK driver: `gps_msgs` with `Customrtk.msg`, and `gps_driver` with `rtk_driver.py`, working on the mosaic-X5 with corrections from RTK2go |
| Lab 3 | Your VN-100 driver, `vn_msgs` and `vn_driver`, working at 40 Hz |
| PA0, PA1 | Ubuntu 24.04 and ROS 2 Jazzy, as before |
| Hardware | One SparkFun RTK mosaic-X5 kit and one VN-100 per team, issued in the classroom, the phone hotspot your receiver joined in Lab 2, an external SSD or USB drive formatted exFAT for copying your car files, and the course's Mustang Mach-E, booked by your team |

## Software to install

```bash
sudo apt install python3-numpy python3-scipy python3-matplotlib python3-ipykernel terminator
```

`terminator` is optional. It splits one window into several terminals, which helps when you watch two topics and a recording at once.

Make sure the course repository is current. The GNSS and IMU emulators, the self-check, and a new RTK test vector of a short Seattle drive are in `~/5554-F26/sensor_emulator`:

```bash
cd ~/5554-F26 && git pull
```

### How to run this notebook

Copy it into your `Lab4/analysis/` folder and use it as the start of your analysis. The analysis code reads rosbags, so open it from a terminal where ROS 2 and your Lab 4 workspace are sourced, and pick the system Python 3.12 as the kernel:

```bash
source /opt/ros/jazzy/setup.bash
source ~/EECE5554/Lab4/ros2_ws/install/setup.bash
code ~/EECE5554/Lab4/analysis
```

A virtual environment hides `rclpy` and `rosbag2_py`, so do not use one for this lab.

## Individual and team work

**Team:** the two walking data sets, the two driving data sets, and the two car files, `car_circles.npz` and `car_drive.npz`. Each teammate commits a copy of all six.
**Individual:** your Lab 4 workspace and launch file, your self-check report, your analysis, your figures, and your answers.

## Collaboration statement

You should complete this assignment without collaboration with other students in the course or elsewhere, beyond the shared hardware and the shared data sets noted above. Please ask for help from the TAs or course instructors if you are getting stuck.

## Safety

The Mustang is a real car on public streets. These rules are not optional.
<ul>
    <li> Only an approved driver drives: [the instructor or the driver in your team] </li>
    <li> The driver only drives. The driver never looks at the laptop, and nobody asks the driver to </li>
    <li> Follow all traffic laws and posted speed limits. Always yield to pedestrians, cyclists, and the streetcar </li>
    <li> Nobody touches a sensor, cable, or mount while the car moves. If something comes loose, keep driving normally until you can stop somewhere legal and safe, and fix it parked </li>
    <li> Secure everything in the cabin before the car moves. In a hard stop, a loose IMU or laptop becomes a projectile </li>
    <li> Nothing goes on the dashboard, near an airbag, or where it blocks the driver's view or the pedals </li>
    <li> Everyone wears a seat belt the whole time </li>
    <li> Anyone can stop the drive if they feel unsafe. A data set can be recollected </li>
</ul>

And remember that not everyone likes robots.

## Additional help

`EECE5554_Lab4_example.md` is a complete worked example: the `nav_bringup` package, a launch file skeleton, a passing self-check report, and the README format. `EECE5554_Lab4_field_handout.pdf` is the field handout: every step at the car, a checklist, the sensor diagram, troubleshooting, and a page for your notes. Print it and keep it in the car, because the car has no internet.

# Part 1: One launch file for both drivers

## Step 1: Build the Lab 4 workspace

Copy your two drivers into a new workspace, and add one package that starts them both:

```bash
mkdir -p ~/EECE5554/Lab4/ros2_ws/src
cd ~/EECE5554/Lab4/ros2_ws/src
cp -r ~/EECE5554/Lab2/ros2_ws/src/gps_msgs ~/EECE5554/Lab2/ros2_ws/src/gps_driver .
cp -r ~/EECE5554/Lab3/ros2_ws/src/vn_msgs ~/EECE5554/Lab3/ros2_ws/src/vn_driver .
ros2 pkg create nav_bringup --build-type ament_python --dependencies gps_driver vn_driver
```

Copy, do not move: your Lab 2 and Lab 3 folders stay as they were graded. If you fix a driver bug during Lab 4, fix it in this workspace.

Target layout:

```
Lab4/ros2_ws/src/
├── gps_msgs/            from Lab 2, unchanged, with Customrtk.msg
├── gps_driver/          from Lab 2, with rtk_driver.py
├── vn_msgs/             from Lab 3, unchanged
├── vn_driver/           from Lab 3
└── nav_bringup/
    ├── launch/
    │   └── nav.launch.py
    ├── nav_bringup/
    │   └── __init__.py
    ├── resource/
    │   └── nav_bringup
    ├── package.xml
    ├── setup.cfg
    └── setup.py
```

**Names matter.** The self-check looks for exactly `nav_bringup` and `nav.launch.py`, for your earlier launch files under the names Lab 2 and Lab 3 gave them, `gps_driver/launch/rtk_driver.launch.py` and `vn_driver/launch/driver.launch.py`, and for `gps_msgs/msg/Customrtk.msg`.

## Step 2: Write the launch file

Create `nav_bringup/launch/nav.launch.py`. It must:

1. Declare two launch arguments, `gps_port` and `imu_port`
2. Include `rtk_driver.launch.py` from `gps_driver`, passing `gps_port` as its `port`
3. Include `driver.launch.py` from `vn_driver`, passing `imu_port` as its `port`

Including your existing launch files, rather than starting the nodes a second way, means each driver keeps the settings you already tested in Lab 2 and Lab 3. This is the pattern for the GPS half:

```python
import os

from ament_index_python.packages import get_package_share_directory
from launch import LaunchDescription
from launch.actions import DeclareLaunchArgument, IncludeLaunchDescription
from launch.launch_description_sources import PythonLaunchDescriptionSource
from launch.substitutions import LaunchConfiguration


def generate_launch_description():
    gps_launch = os.path.join(get_package_share_directory('gps_driver'),
                              'launch', 'rtk_driver.launch.py')
    return LaunchDescription([
        DeclareLaunchArgument('gps_port', default_value='/dev/ttyACM0',
                              description='Serial port of the RTK receiver'),
        IncludeLaunchDescription(
            PythonLaunchDescriptionSource(gps_launch),
            launch_arguments={'port': LaunchConfiguration('gps_port')}.items()),
        # add the imu_port argument and the VN-100 include here
    ])
```

Install `launch/` through `data_files` in `setup.py`, as in Labs 1 to 3. In `package.xml`, next to the two dependencies `ros2 pkg create` added, add `ament_index_python`, `launch`, and `launch_ros` as `exec_depend`. Then:

```bash
cd ~/EECE5554/Lab4/ros2_ws
colcon build --symlink-install
source install/setup.bash
ros2 launch nav_bringup nav.launch.py --show-args
```

`--show-args` lists the launch arguments, and `gps_port` and `imu_port` must be among them.

**Optional, and worth it in the car.** Add `respawn=True` and `respawn_delay=2.0` to the `Node` in each driver's launch file. A driver that dies after a USB hiccup then restarts by itself two seconds later, instead of leaving a hole in your data until somebody notices. A respawn still shows in the launch terminal, so you can see that it happened.

## Step 3: Give each device a name that never changes

The VN-100 shows up as a `/dev/ttyUSB` device and the receiver as a `/dev/ttyACM` device, but the numbers after them depend on what else is plugged in, and they can change after a USB hiccup. The receiver also shows up as more than one `ttyACM` device, and only one of them carries the GGA sentences you set up in Lab 2. Start a driver on the wrong port and it publishes nothing. Linux also gives every USB serial device a name that never changes:

```bash
ls -l /dev/serial/by-id/
```

Each name includes the maker and model of the device. Unplug one device and list again to be sure which name is which. For the receiver, use the port your Lab 2 driver read GGA from. Then launch with those paths:

```bash
ros2 launch nav_bringup nav.launch.py \
    gps_port:=/dev/serial/by-id/<the receiver's GGA port> \
    imu_port:=/dev/serial/by-id/<the VN-100's name>
```

Write both names in the field handout. On a virtual machine, pass both devices through to the guest first. Your Lab 3 USB latency rule still applies to the VN-100: on the laptop you take in the car, check that its latency timer reads 1.

## Step 4: Test with both emulators

Each emulator creates a virtual serial port. Start both, then start both drivers with one command:

```bash
# terminal 1: a short drive through Seattle with RTK, one GNGGA sentence per second
cd ~/5554-F26/sensor_emulator
python3 serial_emulator.py -f data/rtk_seattle.txt -p /tmp/gps_port.txt

# terminal 2: the VN-100 emulator
cd ~/5554-F26/sensor_emulator
python3 imu_emulator.py -f data/imu_stationary.txt -p /tmp/imu_port.txt

# terminal 3, with your workspace sourced
ros2 launch nav_bringup nav.launch.py \
    gps_port:=$(cat /tmp/gps_port.txt) imu_port:=$(cat /tmp/imu_port.txt)
```

The `-p` option tells each emulator where to write its port name. Give them different files: without `-p`, both write `port_file.txt` in the folder you started them from.

In a fourth terminal, `ros2 topic hz /gps` should settle near 1 Hz and `ros2 topic hz /imu` near 40 Hz, and `ros2 topic echo /gps --once` should show zone 10, letter T, and a `fix_quality` of 4 or 5. Then record a short bag with both topics:

```bash
cd /tmp
ros2 bag record -o emulator_test /gps /imu      # Ctrl+C after about 30 s
ros2 bag info emulator_test
```

`ros2 bag info` should list both topics, with about 30 messages on `/gps` and about 1200 on `/imu`.

## Step 5: Run the self-check

```bash
cd ~/5554-F26/sensor_emulator
./verify_lab4.sh <your-northeastern-username>
```

It builds your Lab 4 workspace in a temporary folder and starts both emulators. Then it launches `nav.launch.py` twice, with a different pair of ports each time, and checks that each driver publishes the data from its own device. It also checks your four rosbags and your two car files once they exist, checks your Git hygiene, and writes `~/EECE5554/Lab4/[NortheasternUsername]_lab4.txt`. Every failed check says why.

Run it three times: now, after the dry run, and after the drive. After the dry run, its READY TO DRIVE section must say YES before your team takes the car. After the drive, complete Section 10 of the report and commit it as generated.

# Part 2: The dry run, on foot

Everything that can go wrong in the car can go wrong on foot first. The ports get swapped, a driver dies after three minutes, the receiver never gets RTK corrections, a bag is missing a topic, or your analysis code cannot read the data. On foot, each of those costs you ten minutes. In the car, each one costs the drive, and debugging in a moving car is not safe. So rehearse with the complete car setup, while walking.

Do the dry run as a team, with the laptop you will take in the car, in an open area such as the lawns at Lake Union Park: away from cars, light poles, and metal benches.

## Step 6: Set up exactly as you will in the car

1. One laptop runs both drivers and the recording: the one you will take in the car. Charge it fully
2. Set up the receiver as in Lab 2: the GNSS antenna, the WiFi antenna, the CAT-6 loop between the two Ethernet jacks, and USB-C from CONFIG MOSAIC to the laptop. Turn on the phone hotspot your receiver joined in Lab 2
3. Start `nav.launch.py` with both `/dev/serial/by-id/` names from Step 3
4. Check the rates: `ros2 topic hz /imu` near 40 Hz and `ros2 topic hz /gps` near 1 Hz
5. `ros2 topic echo /gps --once` should show a latitude near 47.6, a longitude near -122.3, zone 10, and letter T. Then wait until `fix_quality` reads 4 (RTK fixed) or 5 (RTK float) before you record. If it stays at 1 for more than a few minutes, corrections are not arriving: check the hotspot, the CAT-6 loop, and the Corrections tab, as in Lab 2
6. Hold the VN-100 level in front of you, with its x axis pointing where you walk, away from your phone and the laptop. A teammate carries the laptop and the receiver, and holds the GNSS antenna up with a clear view of the sky
7. Record with your laptop's WiFi off, as you will in the car: `nmcli radio wifi off`, and `nmcli radio wifi on` afterward. The receiver keeps its corrections through the phone hotspot by itself

## Step 7: Record two walking data sets

| Data set | What to do |
|---|---|
| `walk_circle` | Stand still for 15 s. Walk a circle at least 10 m across, around a landmark, 4 times at a steady pace, holding the VN-100 level with x forward. Stand still for 10 s |
| `walk_square` | Pick a square at least 20 m on a side, with sides north-south and east-west if you can. Stand still at a corner for 15 s. Walk once around at a steady pace, turning sharply through 90 degrees at each corner, and stop where you started. Stand still for 10 s |

```bash
mkdir -p ~/EECE5554/Lab4/data && cd ~/EECE5554/Lab4/data
ros2 bag record -o walk_circle /gps /imu
ros2 bag record -o walk_square /gps /imu
```

One teammate films each walk on a phone, so you can later match features in the data to what happened. Do not commit the videos.

## Step 8: Check before you leave

Run the self-check, or the `check_bag` cell in Part 4, on each bag. Then make Figures 1 to 3 from Part 5.1, the same day. If your analysis code cannot handle the walking data, it will not handle the drive either. Fix whatever failed and record again while you are still there.

READY TO DRIVE in the self-check report says YES when your workspace builds, your launch file passes both emulator rounds, and both walking bags pass, which includes RTK corrections for at least half of each walk. Show it to the instructor when your team picks up the car. Copy the walking bags to every teammate, so each report can say YES, any of your laptops can be the car laptop, and another can be the spare.

# Part 3: The drive

Two computers record each drive. **Your laptop** runs your two drivers and records `/gps` and `/imu`, exactly as in the dry run. **The car PC** records the car's own sensors: the LiDAR on the roof, the LiDAR's built-in IMU, the car's own IMU and GPS, and its wheel speeds. One teammate, the car PC operator, sits at the car PC and runs its recording with `~/lab4/lab4_record.sh`, in a terminal that staff opens. The two computers never connect to each other: the data lead and the car PC operator start and stop their recordings by calling out to each other, and Part 5.6 lines up the two clocks afterward.

The field handout, `EECE5554_Lab4_field_handout.pdf`, has every step of this part in order, with a checklist and a troubleshooting table. Print it: the car has no internet.

## Logistics

| Item | Explain |
|---|---|
| Booking the car | [Ask the Instructor] |
| Where the car is, and how you get the key | [Ask the Instructor] |
| Who may drive | [The Instructor or the driver in your team] |
| Where to drive the calibration circles | [Your reasonable choice] |
| Your team number | [Ask the Instructor]. The car PC files your recordings under it, as `team3` for team 3 |

## Roles

Pick roles before you leave, and write them in the field handout.

| Role | Job |
|---|---|
| Driver | Drives, and nothing else |
| Car PC operator | Runs the car's recording on the car PC when the data lead calls for it, watches its `status`, and at the end makes and copies the car files |
| Data lead | Runs the laptop, calls the start and stop of each recording, watches both topic rates and the fix quality for the whole recording, and says so if one drops |
| Sensor lead | Mounts and checks the VN-100, the antenna, and the receiver, and owns the cables |
| Navigator | Calls the route, keeps the phone hotspot on, notes every stop and anything unusual with the time, and films if your team has a fourth member |

With three people, the sensor lead also navigates.

## Step 9: Plan the route

<ul>
    <li> A loop of 2 to 3 km near Kerry Park, on the flatter streets on top of Queen Anne Hill, that starts and ends at the same spot, with at least 10 turns </li>
    <li> Mostly flat streets, with one hill on purpose. A hill tilts the IMU, and gravity then leaks into the forward accelerometer: a 5 percent grade adds about 0.5 m/s<sup>2</sup>. Part 5 corrects for it, and the LiDAR does not feel gravity at all, so the hill shows the difference. Avoid the steepest streets down the south slope, where that correction would be your largest error </li>
    <li> No tunnels, such as the SR 99 tunnel, no freeways, and no underground garages. Each one loses GNSS </li>
    <li> Open streets help RTK. Tall buildings, trees, and bridges drop it to float or standalone, and the phone needs signal the whole way for the corrections. Some float is fine, and Part 5 shows how to use the fix quality </li>
    <li> A few stops help, such as traffic lights or stop signs. At a stop the speed is known to be zero, and Part 5 uses that </li>
    <li> Give the route to the driver before you start, on paper or on a phone mounted where the driver does not have to touch it </li>
</ul>

## Step 10: Mount the sensors, with the car parked

**The VN-100**
<ul>
    <li> On a flat, rigid surface inside the cabin, such as the floor or the center console, as close to the car's centerline as you can. A seat cushion is not rigid </li>
    <li> Level, checked with a level app on a phone, with its x axis pointing straight ahead along the car </li>
    <li> Fixed so it cannot move when the car brakes hard: strong tape, a strap, or a 3D-printed or cardboard holder that is itself taped down </li>
    <li> Away from speakers, the laptop, phones, and anything magnetic </li>
    <li> Measure where it sits relative to <code>base_link</code>, the center of the rear axle: how far forward (x<sub>c</sub>) and how far to the side (y<sub>c</sub>), in centimeters, with a tape measure. Take a photo. Question 5 needs these numbers </li>
</ul>

**The GNSS antenna.** On the roof, near the middle, screwed onto the antenna mount that stays with the car. Measure where it sits relative to `base_link` too: how far forward (x<sub>a</sub>) and how far to the side (y<sub>a</sub>), in centimeters. At RTK accuracy, Part 5 needs these numbers. Route the antenna cable over the top of a rear door, where the door seal closes on it without crushing it. Never route it through a window someone might close. Tape the cable to the roof so the wind cannot lift it.

**The receiver.** Inside the cabin, on the floor or the rear seat, wedged or strapped so it cannot slide, with the WiFi antenna on and the CAT-6 loop in place. The USB-C cable from CONFIG MOSAIC to the laptop powers it and carries the GGA sentences.

**The phone hotspot.** On, in the car, for the whole drive: the receiver fetches its corrections through it. Use the phone your receiver joined in Lab 2. With any other hotspot, the receiver finds no network.

**Cables.** Tape both USB plugs to the laptop or to a fixed point, so a pulled cable cannot pull a plug out.

**The laptop.** In the data lead's lap or on the rear seat, never on the dashboard, and fully charged.

## Step 11: Check everything before the car moves

1. Turn your laptop's WiFi off for the whole trip: `nmcli radio wifi off`. If your laptop reached the internet in the middle of a recording, Ubuntu could jump its clock, and every timestamp after the jump would be wrong. The receiver keeps its corrections through the phone hotspot by itself
2. Start `nav.launch.py` with your two `/dev/serial/by-id/` names. Open two terminals with `ros2 topic hz /imu` (near 40 Hz) and `ros2 topic hz /gps` (near 1 Hz), and keep both visible for the whole drive
3. `ros2 topic echo /gps --once` must show a real fix, and a `fix_quality` of 4 or 5 before you drive. A parking garage blocks GNSS, so do this outside, and give the corrections a few minutes
4. With the car still, `ros2 topic echo /imu --once` should show a linear acceleration z near -9.8 m/s<sup>2</sup>, because the VN-100's z axis points down
5. On the car PC, after staff has started the car's sensors, the car PC operator runs `~/lab4/lab4_check.sh`. It must end with CAR READY: YES
6. Go through the checklist in the field handout


## Step 12: Record the calibration circles, `car_circles`

Each recording runs on both computers. The car PC's starts first and stops last, so it covers all of yours. The data lead calls each step out loud, and the car PC operator answers when it is done.

At a site of your choice, in an open area with nobody around, away from parked cars, light poles, and buildings:

1. With the car stopped, the car PC operator starts the car's recording (with your team number), and says "car recording". Then the data lead starts yours:

   ```bash
   ~/lab4/lab4_record.sh start team3 car_circles      # on the car PC
   ros2 bag record -o car_circles /gps /imu           # on your laptop, in ~/EECE5554/Lab4/data
   ```

2. Wait 20 s without moving
3. Drive 5 slow, steady circles at walking pace, as tight as is comfortable
4. Stop, and wait 10 s. The data lead stops your recording with Ctrl+C and says "laptop stopped". Then the car PC operator runs `~/lab4/lab4_record.sh stop`. Leave the drivers running

The calibration comes from these circles, so turning a full 360 degrees each time matters more than driving a perfect circle. The same turns also let you line up the two computers' clocks in Part 5.6.

## Step 13: Record the drive, `car_drive`

1. Stopped at the start of your route, start both recordings the same way: `~/lab4/lab4_record.sh start team3 car_drive` on the car PC, then `ros2 bag record -o car_drive /gps /imu` on your laptop
2. Wait 20 s without moving. You will measure the accelerometer bias from these 20 s
3. Drive the loop. The navigator notes each stop and anything unusual, with the time. On the car PC, `~/lab4/lab4_record.sh status` shows the car's recording growing
4. Stop where you started, wait 20 s, and stop both recordings: Ctrl+C on your laptop, then `~/lab4/lab4_record.sh stop` on the car PC
5. If a rate drops to zero during the drive, keep driving normally and finish the loop. Then park somewhere safe, fix the problem, and drive the whole loop again as `car_drive_2`, on both computers. A bag with a hole in it is much harder to analyze
6. A fix quality that drops from 4 to 5 or 1 near buildings is normal. If it stays at 1 for minutes, check the phone hotspot at the next stop

## Step 14: Before you hand the car back

<ul>
    <li> Run <code>ros2 bag info</code> on both of your bags. Both topics must be there, with about 40 IMU messages and 1 GPS message per second of recording </li>
    <li> Run the <code>check_bag</code> cell in Part 4, or the self-check if you have time </li>
    <li> On the car PC, make your two car files, then plug your team's drive into the car PC and copy them. A car bag is about 30 GB, mostly LiDAR and camera; <code>extract</code> keeps what Lab 4 needs in about 10 MB, and <code>copy</code> puts the files in a folder named <code>EECE5554_Lab4_team3</code> on the drive:
<pre>~/lab4/lab4_record.sh extract team3
~/lab4/lab4_record.sh copy team3</pre> </li>
    <li> Back at your laptop, plug in the drive and copy the two files into your repository, with your own drive's name:
<pre>mkdir -p ~/EECE5554/Lab4/data/car
cp /media/$USER/&lt;drive&gt;/EECE5554_Lab4_team3/*.npz ~/EECE5554/Lab4/data/car/</pre>
Give a copy to each teammate </li>
    <li> Take every piece of your equipment out of the car, leave the car as you found it, turn your laptop's WiFi back on with <code>nmcli radio wifi on</code>, and tell the instructor about any problem </li>
</ul>

Staff keeps the full car bags, and Lab 5 uses their camera images. To keep a copy yourself, add `--raw` to `copy`; the drive then needs about 40 GB free.

If your drive fails and you cannot rebook in time, ask the instructor for the course data set. It protects your analysis marks but not your data set marks.

# Part 4: Analysis tools

The cells below read your rosbags and the two car files, and give you a few tools: a quick check of a bag, angle wrapping, and a filter. Run them in order. Cells that need data you have not recorded yet print a reminder instead of failing.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import cumulative_trapezoid
from scipy.signal import butter, sosfiltfilt
from scipy.spatial import cKDTree

### Reading a bag

`read_nav_bag` reads `/imu` and `/gps` from one rosbag folder. Your two drivers stamp their headers from different clocks: the GPS driver from the UTC time in the GGA sentence, and the IMU driver from the laptop's clock. So the reader takes its time axis from the bag instead. `t` is when each message was recorded, by the laptop's clock, in seconds from the first message in the bag. That puts both sensors on one time axis.

The magnetometer comes back in microtesla, as `mag_uT`. `vn_yaw`, `vn_pitch`, and `vn_roll` are the VN-100's own estimates, from the quaternion your driver published, in radians. In `gps`, `quality` is the GGA fix quality your Lab 2 driver published: 4 is RTK fixed, good to a few centimeters, 5 is RTK float, good to tens of centimeters, and 1 is standalone, good to a few meters.

In [ ]:
def gga_time_of_day(sentence):
    """UTC seconds since midnight from the time field of a GGA sentence, or NaN."""
    try:
        f = sentence.split(',')[1]
        return int(f[0:2]) * 3600 + int(f[2:4]) * 60 + float(f[4:])
    except (IndexError, ValueError):
        return np.nan


def quat_to_ypr(q):
    """Yaw, pitch, roll in radians (3-2-1) from quaternions (x, y, z, w), one per row."""
    x, y, z, w = np.asarray(q).T
    yaw = np.arctan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))
    pitch = np.arcsin(np.clip(2 * (w * y - z * x), -1, 1))
    roll = np.arctan2(2 * (w * x + y * z), 1 - 2 * (x * x + y * y))
    return yaw, pitch, roll


def read_nav_bag(path, imu_topic='/imu', gps_topic='/gps'):
    """Read /imu and /gps from one rosbag2 folder into two dicts of NumPy arrays.

    In both dicts, t is when each message was recorded, by the laptop's clock,
    in seconds from the first message in the bag.
    """
    import yaml
    import rosbag2_py
    from rclpy.serialization import deserialize_message
    from rosidl_runtime_py.utilities import get_message

    path = Path(path).expanduser()
    meta = yaml.safe_load((path / 'metadata.yaml').read_text())['rosbag2_bagfile_information']
    reader = rosbag2_py.SequentialReader()
    reader.open(rosbag2_py.StorageOptions(uri=str(path), storage_id=meta['storage_identifier']),
                rosbag2_py.ConverterOptions('cdr', 'cdr'))
    types = {t.name: t.type for t in reader.get_all_topics_and_types()}
    missing = [name for name in (imu_topic, gps_topic) if name not in types]
    if missing:
        raise ValueError(f'{path.name} has no {missing}. Its topics: {sorted(types)}')
    msg_type = {name: get_message(types[name]) for name in (imu_topic, gps_topic)}
    reader.set_filter(rosbag2_py.StorageFilter(topics=[imu_topic, gps_topic]))

    t_imu, t_gps, imu, gps = [], [], [], []
    while reader.has_next():
        topic, data, t_ns = reader.read_next()
        m = deserialize_message(data, msg_type[topic])
        stamp = m.header.stamp.sec + 1e-9 * m.header.stamp.nanosec
        if topic == imu_topic:
            o, w = m.imu.orientation, m.imu.angular_velocity
            a, b = m.imu.linear_acceleration, m.mag_field.magnetic_field
            t_imu.append(t_ns)
            imu.append([stamp, o.x, o.y, o.z, o.w, w.x, w.y, w.z, a.x, a.y, a.z, b.x, b.y, b.z])
        else:
            t_gps.append(t_ns)
            gps.append([stamp, m.latitude, m.longitude, m.altitude, m.utm_easting,
                        m.utm_northing, m.hdop, gga_time_of_day(m.gpgga_read),
                        m.fix_quality, m.num_satellites, m.correction_age])
    if not imu or not gps:
        raise ValueError(f'{path.name}: {len(imu)} IMU and {len(gps)} GPS messages')

    t0 = min(t_imu[0], t_gps[0])        # integer nanoseconds: subtract before converting
    r, g = np.array(imu), np.array(gps)
    yaw, pitch, roll = quat_to_ypr(r[:, 1:5])
    imu = {'t': (np.array(t_imu) - t0) * 1e-9, 'stamp': r[:, 0],
           'gyro': r[:, 5:8], 'accel': r[:, 8:11], 'mag_uT': r[:, 11:14] * 1e6,
           'vn_yaw': yaw, 'vn_pitch': pitch, 'vn_roll': roll}
    gps = {'t': (np.array(t_gps) - t0) * 1e-9, 'stamp': g[:, 0], 'lat': g[:, 1],
           'lon': g[:, 2], 'alt': g[:, 3], 'east': g[:, 4], 'north': g[:, 5],
           'hdop': g[:, 6], 'gga_tod': g[:, 7], 'quality': g[:, 8].astype(int),
           'sats': g[:, 9].astype(int), 'age': g[:, 10]}
    return {'name': path.name, 't0_ns': t0, 'imu': imu, 'gps': gps}

### Checking a bag

`check_bag` prints the numbers the dry run is about: how long you stood still at the start, the rate and the largest gap of each topic, how many GPS messages have a fix, how much of the time the fix was RTK fixed or float, how long after each GPS fix the laptop recorded it, how far the gyro turned, and how far the GPS track went. Run it on every bag before you leave the site.

In [ ]:
def check_bag(bag):
    """Print the numbers the dry run is about. Run it on every bag before you leave the site."""
    imu, gps, name = bag['imu'], bag['gps'], bag['name']
    duration = max(imu['t'][-1], gps['t'][-1])
    n_imu, n_gps = imu['t'].size, gps['t'].size
    imu_hz = (n_imu - 1) / np.ptp(imu['t'])
    gps_hz = (n_gps - 1) / np.ptp(gps['t'])
    imu_gap, gps_gap = np.diff(imu['t']).max(), np.diff(gps['t']).max()
    fix = (np.abs(gps['lat']) > 1e-6) & (np.abs(gps['lon']) > 1e-6)

    # how long after each fix the laptop recorded it: laptop time minus the GGA time
    recorded = (bag['t0_ns'] * 1e-9 + gps['t']) % 86400
    delay = (recorded - gps['gga_tod'] + 43200) % 86400 - 43200
    turned = np.degrees(cumulative_trapezoid(imu['gyro'][:, 2], imu['t'])[-1])
    e = gps['east'][fix] - gps['east'][fix][0]
    n = gps['north'][fix] - gps['north'][fix][0]

    # still at the start: until the gyro turns, the accelerometer shakes, or GPS moves 4 m
    smooth = lambda v: np.convolve(np.pad(v, (20, 19), mode='edge'), np.ones(40) / 40, mode='valid')
    a = np.linalg.norm(imu['accel'], axis=1)
    shake = np.sqrt(np.maximum(smooth(a * a) - smooth(a) ** 2, 0))
    moving = (smooth(np.linalg.norm(imu['gyro'], axis=1)) > 0.05) | (shake > 0.1)
    moved = np.hypot(e, n) > 4.0
    still = min(imu['t'][np.argmax(moving)] if moving.any() else duration,
                gps['t'][fix][np.argmax(moved)] if moved.any() else duration)

    print(f'{name}: {duration:.1f} s, still for the first {still:.0f} s')
    print(f'  /imu {n_imu:6d} messages  {imu_hz:5.1f} Hz  largest gap {imu_gap:.2f} s')
    print(f'  /gps {n_gps:6d} messages  {gps_hz:5.2f} Hz  largest gap {gps_gap:.1f} s'
          f'  with a fix {fix.sum()} of {n_gps}  median HDOP {np.median(gps["hdop"]):.1f}')
    q = gps['quality']
    shares = ', '.join(f'{label} {100 * np.mean(q == k):.0f}%' for k, label in
                       ((4, 'RTK fixed'), (5, 'RTK float'), (1, 'standalone')))
    print(f'  fix quality: {shares}')
    print(f'  GPS fix to recording: median {np.nanmedian(delay):.2f} s,'
          f' spread {np.nanpercentile(delay, 95) - np.nanpercentile(delay, 5):.2f} s')
    print(f'  turned {turned:+.0f} degrees about gyro z ({turned / 360:+.1f} circles)')
    print(f'  GPS farthest from the start {np.hypot(e, n).max():.0f} m,'
          f' end to start {np.hypot(e[-1], n[-1]):.0f} m')

### Angles and filters

`wrap` brings angles back into [-π, π). `butter_filter` is a zero-phase Butterworth filter: it does not delay the signal, which suits analysis after the fact. It pads each end by one period of the cutoff, so the ends of a recording settle like the middle.

In [ ]:
def wrap(angle):
    """Wrap angles in radians to [-pi, pi)."""
    return (np.asarray(angle) + np.pi) % (2 * np.pi) - np.pi


def butter_filter(x, cutoff_hz, fs, kind='low', order=1):
    """Zero-phase Butterworth filter. kind is 'low' or 'high'; fs is the sample rate in Hz."""
    if not 0 < cutoff_hz < fs / 2:
        raise ValueError(f'cutoff {cutoff_hz} Hz must be between 0 and fs/2 = {fs / 2} Hz')
    sos = butter(order, cutoff_hz, btype=kind, fs=fs, output='sos')
    return sosfiltfilt(sos, x, padlen=min(len(x) - 1, int(fs / cutoff_hz)))

The complementary filter in Part 5.2 depends on one property: a low pass and a high pass with the same cutoff add back up to the signal. Check it on a wandering signal:

In [ ]:
rng = np.random.default_rng(1)
fs_demo = 40.0
x = 300 + np.cumsum(rng.normal(size=4000))      # any slowly wandering signal
low = butter_filter(x, 0.05, fs_demo, 'low')
high = butter_filter(x, 0.05, fs_demo, 'high')
print(f'signal range {np.ptp(x):.0f}, largest |low + high - signal| {np.abs(low + high - x).max():.3f}')

Close, but not exact at the ends of the recording. Defining the high pass as the signal minus its low pass makes the two halves add up exactly, so Part 5.2 does that.

### A complementary filter, on a heading you know

Five synthetic minutes in which the true heading turns 90 degrees every minute. The "magnetometer" sees the true heading plus a slow wobble and noise, but it never drifts. The "gyro" sees the true turn rate plus a small bias, so its integral drifts. Low pass the magnetometer, high pass the gyro with the same cutoff, add the two, and compare all three with the truth. Then change `fc` and watch what happens to the error.

In [ ]:
t = np.arange(0, 300, 1 / fs_demo)
true_rate = np.where((t % 60) < 6, np.radians(90) / 6, 0.0)      # a 90 degree turn every minute
true_yaw = cumulative_trapezoid(true_rate, t, initial=0)
rng = np.random.default_rng(0)
mag_yaw = wrap(true_yaw + np.radians(6) * np.sin(2 * np.pi * t / 23)
               + rng.normal(0, np.radians(3), t.size))
gyro_yaw = mag_yaw[0] + cumulative_trapezoid(true_rate + np.radians(0.05)
                                             + rng.normal(0, 0.003, t.size), t, initial=0)

fc = 0.005    # Hz. Try 0.002, 0.02, and 0.2
mag_unwrapped = gyro_yaw + wrap(mag_yaw - gyro_yaw)      # unwrap against the gyro
low = butter_filter(mag_unwrapped, fc, fs_demo, 'low')
high = gyro_yaw - butter_filter(gyro_yaw, fc, fs_demo, 'low')
fused = low + high

rms = lambda y: np.degrees(np.sqrt(np.mean(wrap(y - true_yaw) ** 2)))
fig, ax = plt.subplots(figsize=(9, 4))
for y, label in ((mag_yaw, 'magnetometer'), (gyro_yaw, 'gyro'), (fused, f'fused, fc = {fc} Hz')):
    ax.plot(t, np.degrees(wrap(y - true_yaw)), lw=0.8, label=f'{label}: RMS {rms(y):.1f} deg')
ax.set_xlabel('time (s)')
ax.set_ylabel('heading error (deg)')
ax.set_title('Synthetic heading: the error of each estimate')
ax.grid(alpha=0.3)
ax.legend()
plt.show()

### Your data

The cell below reads each of your four bags that exists and checks it.

In [ ]:
DATA = Path('~/EECE5554/Lab4/data').expanduser()

bags = {}
for name in ('walk_circle', 'walk_square', 'car_circles', 'car_drive'):
    if (DATA / name / 'metadata.yaml').exists():
        bags[name] = read_nav_bag(DATA / name)
        check_bag(bags[name])
    else:
        print(f'{name}: not recorded yet (Part 2 and Part 3)')

### Reading the car files

`car_circles.npz` and `car_drive.npz` come from `lab4_record.sh extract` on the car PC, in Step 14. `read_car_npz` reads one into dictionaries like those of `read_nav_bag`. Here `t` is when each message was recorded by the **car PC's** clock, in seconds from the first message in the car's bag. That is not your laptop's clock, and Part 5.6 finds the difference.

| Key | What it holds |
|---|---|
| `lidar_imu` | The Ouster's built-in IMU, 100 Hz: `gyro` in rad/s and `accel` in m/s<sup>2</sup> |
| `car_imu` | The car's own IMU, 100 Hz, with the same fields |
| `car_gps` | The car's own GPS, 1 Hz: `lat`, `lon`, and `alt` |
| `gps_time` | The car GPS's time, 1 Hz: `stamp` is when the car PC received it and `utc` is the time it reports, both in seconds since 1970 |
| `vehicle` | `wheel_speeds`, `vehicle_velocity`, and `steering`: every number in those messages, by name. `check_car` lists them |
| `scans` | The LiDAR, 2 scans per second: `xy[k]` is scan k, the points 1 to 3 m above the ground, as x and y in meters |

**Frames.** The scans and both IMUs here use the ROS convention for vehicles: x forward, y to the left, and z up. The scans are already moved to `base_footprint`, the center of the rear axle on the ground. Your VN-100 uses x forward, y to the right, and z down, so a right turn is a positive gyro z on the VN-100 and a negative one on the Ouster.

In [ ]:
def read_car_npz(path):
    """Read one car file, made on the car PC in Step 14, into dictionaries of NumPy arrays."""
    path = Path(path).expanduser()
    z = np.load(path, allow_pickle=False)
    groups = {}
    for key in z.files:
        if '__' in key:
            group, field = key.split('__', 1)
            groups.setdefault(group, {})[field] = z[key]
    car = {'name': path.stem, 't0_ns': int(z['t0_ns']), 'notes': json.loads(str(z['notes']))}
    for g in ('lidar_imu', 'car_imu', 'car_gps', 'gps_time'):
        car[g] = groups.get(g, {})
    car['vehicle'] = {g: groups[g] for g in ('wheel_speeds', 'vehicle_velocity', 'steering') if g in groups}
    s = groups.get('scans', {})
    off = s.get('offsets', np.zeros(1, dtype=int))
    car['scans'] = {'t': s.get('t', np.array([])), 'stamp': s.get('stamp', np.array([])),
                    'xy': [s['xy'][off[k]:off[k + 1]].astype(float) for k in range(len(off) - 1)]}
    return car


def check_car(car):
    """Print what a car file holds, as check_bag does for your laptop bags."""
    print(f"{car['name']}: car file")
    for g in ('lidar_imu', 'car_imu', 'car_gps', 'gps_time'):
        t = car[g].get('t')
        if t is None or len(t) < 2:
            print(f'  {g:16s} missing')
        else:
            print(f'  {g:16s} {len(t):7d} messages {(len(t) - 1) / np.ptp(t):6.1f} Hz  over {np.ptp(t):6.1f} s')
    n = np.array([len(x) for x in car['scans']['xy']])
    if n.size > 1:
        print(f"  {'scans':16s} {n.size:7d} scans    {(n.size - 1) / np.ptp(car['scans']['t']):6.1f} Hz  "
              f'{np.median(n):.0f} points each (median), fewest {n.min()}')
    else:
        print(f"  {'scans':16s} missing")
    for name, d in car['vehicle'].items():
        fields = [k for k in d if k != 't' and not k.startswith('header.')]
        print(f"  {name:16s} {len(d['t']):7d} messages, fields: {', '.join(fields)}")
    for note in car['notes']:
        print(f'  note: {note}')

In [ ]:
cars = {}
for name in ('car_circles', 'car_drive'):
    path = DATA / 'car' / f'{name}.npz'
    if path.exists():
        cars[name] = read_car_npz(path)
        check_car(cars[name])
    else:
        print(f'{name}.npz: not copied yet (Step 14)')

# Part 5: Analysis

Thirteen figures, a results table, and twelve questions. The tools above read your data, and everything after them is yours to write. Every figure needs axis labels with units, a legend wherever there is more than one line, and time that starts at 0. Plots of magnetometer readings and of tracks need equal axes: `ax.set_aspect('equal')`.

## 5.1 The dry run (Figures 1 to 3)

Do this part before your drive. The same code then runs on the car data.

### Magnetometer calibration

A level magnetometer turned through a full circle should trace a circle centered on zero, with a radius equal to the horizontal part of the Earth's field, about 19 µT in Seattle. Real readings trace an ellipse that is off center, because of iron that moves with the sensor:
<ul>
    <li> <b>Hard iron.</b> Magnetized material, such as a speaker magnet, a magnetized steel part, or a phone, adds a constant field. It shifts the circle </li>
    <li> <b>Soft iron.</b> Unmagnetized iron bends the Earth's field toward itself. It stretches and tilts the circle into an ellipse </li>
</ul>

A calibration undoes both: $m_{cal} = S\,(m - c)$, where $c$ is the center of the ellipse (the hard iron) and $S$ is a 2 × 2 matrix that turns the ellipse back into a circle (the soft iron). Use the x and y readings only.

Do not use `magcal` or any other calibration function. Write the fit yourself, with NumPy:

1. Stack one row $[x^2,\ xy,\ y^2,\ x,\ y,\ 1]$ per reading into a matrix $D$. Readings on an ellipse satisfy $Dq = 0$ for the ellipse's six coefficients $q = (q_1, \dots, q_6)$. The least-squares $q$ with $|q| = 1$ is the last row of $V^T$ from `np.linalg.svd(D)`
2. The center solves $\begin{bmatrix} 2q_1 & q_2 \\ q_2 & 2q_3 \end{bmatrix} c = -\begin{bmatrix} q_4 \\ q_5 \end{bmatrix}$
3. With $Q = \begin{bmatrix} q_1 & q_2/2 \\ q_2/2 & q_3 \end{bmatrix}$, every reading on the ellipse satisfies $(m - c)^T M (m - c) = 1$, where $M = Q \,/\, (c^T Q c - q_6)$
4. $S = r\,M^{1/2}$, where $M^{1/2}$ is the symmetric square root, built from the eigenvalues and eigenvectors that `np.linalg.eigh(M)` returns. Then $S(m - c)$ lies on a circle of radius $r$. With $r = \det(M)^{-1/4}$, the circle keeps the area of the ellipse

If an eigenvalue of $M$ is not positive, the fit is not an ellipse: the data did not turn through a full circle, or the sensor moved during the circles.

In [ ]:
def fit_ellipse(x, y):
    """Fit the ellipse traced by raw magnetometer x and y, and the map that makes it a circle.

    Returns (center, soft, radius): soft @ (m - center) lies on a circle of the
    given radius for every raw reading m = (x, y) on the ellipse.
    """
    raise NotImplementedError('Write fit_ellipse here (Part 5.1)')


def calibrate(mag_xy, center, soft):
    """Apply a hard- and soft-iron calibration to raw x, y readings, one per row."""
    return (np.asarray(mag_xy) - center) @ np.asarray(soft).T

Check your fit on an ellipse whose answer you know before you trust it on your data. `test_ellipse` makes readings with a hard iron of (12, -7) µT, a soft iron that stretches one axis by 25 percent and squeezes the other by 20 percent, tilted by 30 degrees, and a little noise.

In [ ]:
def test_ellipse(seed=0):
    """Raw magnetometer x, y on a known ellipse, to check fit_ellipse."""
    rng = np.random.default_rng(seed)
    th = np.linspace(0, 6 * np.pi, 600)
    circle = 20.0 * np.stack([np.cos(th), np.sin(th)], axis=1)
    c, s = np.cos(np.radians(30)), np.sin(np.radians(30))
    rot = np.array([[c, -s], [s, c]])
    soft = rot @ np.diag([1.25, 0.8]) @ rot.T
    raw = circle @ soft.T + np.array([12.0, -7.0]) + rng.normal(0, 0.3, circle.shape)
    return raw, {'center': np.array([12.0, -7.0]), 'soft': soft, 'radius': 20.0}


raw, truth = test_ellipse()
try:
    center, soft, radius = fit_ellipse(raw[:, 0], raw[:, 1])
    r = np.hypot(*calibrate(raw, center, soft).T)
    print('center found', center.round(2), '  true', truth['center'])
    print(f'calibrated radius {r.mean():.2f} microtesla, spread {r.std():.2f} (the noise alone gives about 0.3)')
    print('your soft matrix times the true soft iron, which is close to the identity if your fit is right:')
    print((soft @ truth['soft']).round(3))
except NotImplementedError as err:
    print(err)

### Heading from the magnetometer and from the gyro

With the VN-100 level, x forward and y to the right, the heading clockwise from magnetic north is

$$\psi_{mag} = \operatorname{atan2}(-m_y,\ m_x)$$

using calibrated readings. Check the sign on your own data: on the square walk, each right turn should add about 90 degrees.

In the VN-100 frame z points down, so a positive gyro z is a turn to the right, the same direction in which heading grows. Integrate it, starting from the magnetometer heading while you stood still:

$$\psi_{gyro}(t) = \psi_{mag}(0) + \int_0^t \omega_z\,dt$$

`cumulative_trapezoid(imu['gyro'][:, 2], imu['t'], initial=0)` does the integral.

Magnetic north is not true north. In Seattle it is about 15 degrees east of true north. Look up the declination for your location and date with NOAA's calculator, https://www.ngdc.noaa.gov/geomag/calculators/magcalc.shtml. Heading from true north is $\psi_{mag}$ plus the declination.

### Dead reckoning on foot

Walking jolts the accelerometer far too much to integrate it twice, so on foot take your speed from GPS. You walked at a steady pace, so use the average:

$$\bar v = \frac{\text{distance walked, from GPS}}{\text{time spent walking}}$$

GPS noise adds a little distance at every fix, much less with RTK fixed than without, so count only moves of half a meter or more. Then integrate the velocity into east and north, with $\bar v$ while you walked and 0 while you stood still:

$$v_E = \bar v \sin\psi \qquad v_N = \bar v \cos\psi$$

Rotate your track by the declination before you compare it with the RTK track. UTM east and north follow the map grid, which in Seattle is within half a degree of true north.

### Figures 1 to 3

| Figure | Content |
|---|---|
| 1 | `walk_circle`: magnetometer x against y, raw and calibrated, on equal axes, with your fitted ellipse drawn on the raw data |
| 2 | `walk_square`: heading against time from the raw magnetometer, the calibrated magnetometer, and the gyro, unwrapped, in degrees |
| 3 | `walk_square`: east against north in meters from the start, from the RTK track and from dead reckoning with each of your two headings, on equal axes |

## 5.2 Heading in the car (Figures 4 to 7)

Calibrate from `car_circles` exactly as you did in 5.1, and apply that calibration to `car_drive`. Do not reuse the walking calibration: the car's steel body, motors, and electronics are a different magnetic environment, and they move with the sensor.

**The complementary filter.** The magnetometer is right on average, but it jumps: every steel structure, passing truck, and change in the car's electric current moves it. The gyro is smooth, but it drifts. So keep the slow part of one and the fast part of the other, with the same cutoff frequency $f_c$ for both:

$$\psi_{cf} = \text{LPF}(\psi_{mag}) + \text{HPF}(\psi_{gyro}) \qquad \text{HPF}(\psi_{gyro}) = \psi_{gyro} - \text{LPF}(\psi_{gyro})$$

Defining the high pass that way makes the two halves add up exactly, as the check in Part 4 showed. Three things to get right:

- Unwrap the magnetometer heading before you filter it, or every pass through ±180 degrees becomes a 360 degree spike. Unwrapping against the gyro is robust: $\psi_{gyro} + \text{wrap}(\psi_{mag} - \psi_{gyro})$, as in the demo in Part 4
- $f_c$ sets a time scale of about $1/(2\pi f_c)$ seconds, beyond which you trust the magnetometer over the gyro. Try values from 0.002 to 0.2 Hz, look at the results, and justify your choice in Question 2
- Wrap the result back to [-180, 180) degrees before you compare it with the VN-100 yaw

The VN-100 computes its own yaw with its own filter. You may use it in exactly one place: the last panel of Figure 7, to compare with yours. Nowhere else.

| Figure | Content |
|---|---|
| 4 | `car_circles`: magnetometer x against y, raw and calibrated, on equal axes, with your fitted ellipse drawn on the raw data |
| 5 | `car_drive`: magnetometer heading against time, before and after calibration |
| 6 | `car_drive`: gyro heading against time, with the calibrated magnetometer heading on the same axes |
| 7 | `car_drive`, four panels against time: the LPF of the magnetometer heading, the HPF of the gyro heading, your complementary filter, and the VN-100's yaw |

## 5.3 Forward velocity (Figure 8)

Integrate the forward acceleration, `imu['accel'][:, 0]`, to get forward velocity. The first attempt will be wrong. The point is to find out why, and fix it:

- **Bias.** The accelerometer reads a little even when the car is still. Measure it during the first 20 s, while the car stood still, and subtract it
- **Hills.** The accelerometer measures gravity too. With the nose up by a pitch $\theta$, the x axis reads $a_x + g\sin\theta$, so a 3 percent grade looks like 0.3 m/s<sup>2</sup> of acceleration for as long as the hill lasts. `imu['vn_pitch']` is the VN-100's estimate of $\theta$. You may use its pitch and roll, but not its yaw
- **Stops.** At every stop the true speed is zero. Find the stops (the gyro and the accelerometer go quiet, and the GPS speed drops to zero), set the velocity to zero there, and remove the drift that built up since the last stop
- **Common sense.** A car driving a loop forward never goes backward

Then compute speed from the RTK fixes: the distance between consecutive UTM positions, divided by the time between them. With RTK fixed, that speed is good to a few centimeters per second. Float and standalone fixes are much noisier, so mark them.

The car measures its own speed too. `cars['car_drive']['vehicle']` holds its wheel speeds and its velocity estimate, and `check_car` lists their fields. Find the one that is the car's forward speed, work out its units by comparing it with the RTK speed on a straight stretch, and put it on your laptop's time with the $L$ from Part 5.6.

| Figure | Content |
|---|---|
| 8 | Two panels against time: (a) the accelerometer velocity before any adjustment; (b) after your adjustments, with the RTK speed and the car's own speed on the same axes |

## 5.4 A motion model for the car (Figure 9)

Model the car as a rigid body on a flat road, with tires that do not slide sideways. Then one point always moves straight ahead: `base_link`, the center of the rear axle. Call its forward speed $V$ and the car's turn rate $\omega$. In the VN-100 frame, with z down, $\omega$ is gyro z, and it is positive in a right turn. The IMU sits at $r = (x_c, y_c)$ from `base_link`, in the car's frame.

`base_link` moves along the car's x axis, and that axis turns at $\omega$. So in the car's frame its acceleration has two parts: $\dot V$ along x, and the centripetal acceleration $\omega V$ across:

$$a_{base} = (\dot V,\ \omega V)$$

A point fixed on a turning body adds two terms of its own, one from the change in the turn rate and one from the turn itself:

$$a_{IMU} = a_{base} + \dot\omega \times r + \omega \times (\omega \times r)$$

In components, this is what a level accelerometer at the IMU reads on a flat road:

$$\ddot x_{obs} = \dot V - \omega^2 x_c - \dot\omega\, y_c \qquad \ddot y_{obs} = \omega V + \dot\omega\, x_c - \omega^2 y_c$$

With the IMU on the centerline, $y_c = 0$:

$$\ddot x_{obs} = \dot V - \omega^2 x_c \qquad \ddot y_{obs} = \omega V + \dot\omega\, x_c$$

So the lateral accelerometer reads $\omega V$, plus a term that appears only while the turn rate changes, on the way into and out of each turn. That term lets you find $x_c$ from the data alone: there, $\ddot y_{obs} - \omega V \approx \dot\omega\, x_c$.

What to do:
1. Compute $\omega V$ from gyro z and your adjusted velocity from 5.3
2. Low pass filter both $\ddot y_{obs}$, which is `imu['accel'][:, 1]`, and $\omega V$ at about 1 Hz, so road bumps do not hide the comparison
3. Plot both on the same axes, for the whole drive and zoomed in on one turn (Figure 9)
4. Estimate $x_c$ by least squares from $\ddot y_{obs} - \omega V = \dot\omega\, x_c$, with $\dot\omega$ from your filtered gyro z, using only the samples where the car moves. Compare it with your tape measurement in Question 5

Sign check: in a right turn, $\omega > 0$, and the centripetal acceleration points right, along +y. Both curves are positive. If yours have opposite signs, check the frame, not the physics.

| Figure | Content |
|---|---|
| 9 | $\ddot y_{obs}$ and $\omega V$ against time, low pass filtered, on the same axes: the whole drive, and a zoom on one turn |

## 5.5 Dead reckoning (Figure 10)

Rotate your forward speed into east and north with your best heading, the complementary filter or another one you trust more, and integrate both into a track $(x_E, x_N)$:

$$v_E = V \sin\psi \qquad v_N = V \cos\psi$$

Then compare it with the RTK track, in east and north from the first fix:
1. Move the RTK track to `base_link`. The receiver reports where the antenna is, and the antenna sits $x_a$ ahead of `base_link` and $y_a$ to its right. At RTK accuracy that offset matters, most of all in turns. Subtract $x_a(\sin\psi,\ \cos\psi) + y_a(\cos\psi,\ -\sin\psi)$ from each fix's (east, north), where $\psi$ is the car's heading from true north at that time: your heading plus the declination
2. Shift your track so that both start at the first RTK fix
3. Rotate your track about its start so that its first straight stretch points the same way as the RTK track. Report the angle. Most of it should be the magnetic declination, and the rest is how straight the IMU was mounted
4. Compare the lengths of the two tracks, and report their ratio as your scale factor. Report it rather than quietly applying it
5. Compute the distance between your track and the RTK track at every fix, and keep each fix's quality with it

| Figure | Content |
|---|---|
| 10 | (a) the RTK and dead-reckoned tracks, east against north, on equal axes, with the start marked and the RTK track colored by fix quality; (b) the distance between them against time since the car started moving, with lines at 2 m and 10 m |

## 5.6 Two clocks (Figure 11)

Your laptop and the car PC each timed their own bag, by their own clock, and neither clock was synchronized during the drive, because both were offline. So 10 s into your `car_drive` bag and 10 s into the car's are not the same moment. Call the difference $L$: a moment that your bag records at time $t$, the car's bag records at $t + L$. $L$ is roughly how much earlier the car's recording started, but a start time typed by hand is good to a second at best. Measure it from the data instead.

Both computers recorded a gyro fixed to the same car: your VN-100 and the Ouster's built-in IMU. Every turn shows up in both at the same physical moment. Slide one gyro's record in time until it matches the other's: the slide that matches best is $L$.

1. Take gyro z from both: `bags['car_circles']['imu']['gyro'][:, 2]` and `cars['car_circles']['lidar_imu']['gyro'][:, 2]`. The VN-100's z axis points down and the Ouster's points up, so flip the sign of one. Check on the circles: after the flip, both curves have the same sign in every turn
2. Write `find_lag` below. Resample the laptop's gyro on a regular grid at 100 Hz. For each candidate $L$ from $-60$ to $60$ s, in steps of 0.01 s, interpolate the car's gyro at the grid times plus $L$, and compute the correlation coefficient between the two: subtract each one's mean, then divide their dot product by both of their lengths. Keep the $L$ with the highest correlation. A parabola through that peak and its two neighbors gives $L$ to about a millisecond
3. Run it on `car_circles`, where the turns are large and steady, and on `car_drive`. A correlation above 0.95 means the match is real. Near 0.3 usually means a sign was not flipped, or one curve is not gyro z
4. Run it on the first half and on the second half of `car_drive`. The two values differ a little, because no two clocks tick at exactly the same rate. Their difference divided by the time between the two halves' centers is the drift, in parts per million (ppm)
5. Check $L$ with GPS time. `check_bag` already prints how late your laptop recorded each GGA sentence: that is your laptop's clock minus UTC, plus a small delay. `stamp - utc` in `cars['car_drive']['gps_time']` is the same for the car PC. With $t_{0,lap}$ and $t_{0,car}$ the two bags' `t0_ns` in seconds, $L_{gps} = (t_{0,lap} - t_{0,car}) + (\text{car PC} - \text{UTC}) - (\text{laptop} - \text{UTC})$, using the median of each difference. It should agree with your gyro $L$ to within a few tenths of a second; the rest is how late each GPS receiver reports its time
6. From here on, put every car time on your laptop's time: $t_{laptop} = t_{car} - L$. Use the $L$ from `car_drive` for the drive, and the one from `car_circles` for the circles

| Figure | Content |
|---|---|
| 11 | (a) gyro z from both IMUs against your laptop's time, after the shift by $L$ and the sign flip, for one minute of `car_circles`; (b) the correlation against $L$ for `car_drive`, with its peak marked |

In [ ]:
def find_lag(t_lap, w_lap, t_car, w_car, max_lag=60.0, rate=100.0):
    """L such that a moment at time t in the laptop bag is at time t + L in the car bag.

    w_lap and w_car are the same turn rate, with the same sign, from the two bags.
    Returns (L, best correlation, the candidate lags, the correlation at each lag).
    """
    raise NotImplementedError('Write find_lag here (Part 5.6)')

Check it on two gyros whose lag you know before you trust it on your data. `test_lag` makes a car gyro at 100 Hz with z up, and a laptop gyro at 40 Hz with z down whose recording started 7.35 s later. With the sign flipped, your lag should be within a few milliseconds of 7.35 s, with a correlation near 0.99. Without the flip, the correlation collapses: that is the warning sign to look for on your own data.

In [ ]:
def test_lag(seed=0, lag=7.35):
    """A car gyro (100 Hz, z up) and a laptop gyro (40 Hz, z down) that started recording lag s later."""
    rng = np.random.default_rng(seed)
    t = np.arange(0, 300, 0.01)
    rate = np.where((t % 40) < 8, np.radians(45) / 8 * np.sign(np.sin(2 * np.pi * t / 160) + 0.1), 0.0)
    w_car = rate + rng.normal(0, 0.005, t.size)
    t_lap = np.arange(0, 300 - lag, 1 / 40)
    w_lap = -np.interp(t_lap + lag, t, rate) + rng.normal(0, 0.005, t_lap.size)
    return t_lap, w_lap, t, w_car, lag


t_lap, w_lap, t_car, w_car, true_lag = test_lag()
try:
    L, corr, _, _ = find_lag(t_lap, -w_lap, t_car, w_car, max_lag=30)
    print(f'with the sign flipped: lag {L:.3f} s, true {true_lag} s, correlation {corr:.3f}')
    L, corr, _, _ = find_lag(t_lap, w_lap, t_car, w_car, max_lag=30)
    print(f'without the flip:      lag {L:.3f} s, correlation {corr:.3f}')
except NotImplementedError as err:
    print(err)

## 5.7 LiDAR odometry (Figure 12)

The accelerometer measures acceleration, and integrating it twice turns every small error into a large one. The LiDAR measures the world around the car. If two scans taken half a second apart see the same walls, trees, and poles, the turn and shift that line one scan up with the other is how the car moved in that half second. Chaining those steps gives the car's path. That is LiDAR odometry, and the way to line up two scans is ICP, iterative closest point.

**ICP for two scans.** Given a scan `src`, an earlier scan `dst`, both $n \times 2$ arrays of points, and a guess of the turn $\psi_0$ and shift $t_0$ that carry `src` onto `dst`:

1. Build a KD-tree of `dst` once: `tree = cKDTree(dst)`
2. Move `src` by the current estimate: $p_i = R\,s_i + t$, with $R$ the rotation by the current turn
3. Find each moved point's nearest neighbor in `dst` with `tree.query(p, distance_upper_bound=d_max)`, and drop the pairs farther apart than $d_{max}$. Start with $d_{max} = 2$ m and shrink it to 0.5 m over the iterations, so pairs that do not belong together cannot pull the answer
4. Find the best turn and shift for those pairs. Subtract each set's centroid, $\bar p$ and $\bar q$, form $H = \sum_i (p_i - \bar p)(q_i - \bar q)^T$, take `U, S, Vt = np.linalg.svd(H)`, and set $R_\Delta = V U^T$. If $\det R_\Delta < 0$, flip the sign of the last row of $V^T$ and form it again. Then $t_\Delta = \bar q - R_\Delta \bar p$
5. Update the estimate, $R \leftarrow R_\Delta R$ and $t \leftarrow R_\Delta t + t_\Delta$, and repeat from step 2 until the update is tiny, or for 40 iterations
6. Return the turn $\psi = \operatorname{atan2}(R_{10}, R_{00})$, the shift $t$, the RMS distance of the final pairs, and the share of points that found a pair

**The guess matters.** Along a street, the walls look the same for tens of meters, so ICP started from no motion at all can slide along the street and stop in the wrong place. Give it a good start: for the turn, the Ouster's gyro z over the half second, with its bias from the first 20 s removed; for the shift, the car's own speed from 5.3 times the half second, along x.

**Chaining.** Each step gives the pose of scan $k$ in the frame of scan $k-1$. Starting from $(x, y, a) = (0, 0, 0)$:

$$x_k = x_{k-1} + t_x \cos a_{k-1} - t_y \sin a_{k-1} \qquad y_k = y_{k-1} + t_x \sin a_{k-1} + t_y \cos a_{k-1} \qquad a_k = a_{k-1} + \psi$$

That track is in the car's starting frame: x forward, y to the left, and angles counterclockwise. With the car's starting heading $\psi_0$, clockwise from true north, east is $x \sin\psi_0 - y\cos\psi_0$ and north is $x\cos\psi_0 + y\sin\psi_0$. Take $\psi_0$ from the RTK track's first straight stretch, as in step 3 of 5.5, start the track at the first RTK fix moved to `base_link`, and put the scan times on your laptop's time with $L$ from 5.6.

What to do:

1. Write `icp_2d` below, and check it on the test pair
2. Run it on every pair of consecutive scans in `car_drive`, chain the steps, and keep each step's RMS and pair share. Steps with a share under 0.5 are suspect: find where they happened
3. Compare the track with the RTK track exactly as you compared your dead reckoning in 5.5: the distance at every fix, the time within 2 m and within 10 m, and the distance at the end of the loop, also as a share of the distance driven

| Figure | Content |
|---|---|
| 12 | (a) the RTK track, your dead reckoning from 5.5, and your LiDAR odometry, east against north, on equal axes; (b) the distance from the RTK track against time for both estimates, with lines at 2 m and 10 m |

In [ ]:
def icp_2d(src, dst, yaw0=0.0, t0=(0.0, 0.0), iters=40):
    """Align scan src onto scan dst, both (n, 2) arrays, starting from the guess yaw0 and t0.

    Returns (yaw in radians, t as a length 2 array, RMS distance of the final pairs in m,
    share of src points that found a pair), such that R(yaw) @ s + t lands on dst.
    """
    raise NotImplementedError('Write icp_2d here (Part 5.7)')

Check it on two scans of a street whose answer you know. `test_scan_pair` makes two scans with two walls, two cross streets, a wall at an angle, and twelve poles. The second scan is taken after the car turned 4 degrees left and moved 3.0 m forward and 0.4 m left. Each scan hits the surfaces at its own random spots, as a real LiDAR does, so the RMS never reaches zero: about 0.1 m is normal.

With a good guess, your answer should land within a few centimeters and a few hundredths of a degree. With no guess at all, the turn and the sideways shift still come out right, but the forward shift does not: along the street, the walls look the same everywhere.

In [ ]:
def test_scan_pair(seed=0, turn_deg=4.0, shift=(3.0, 0.4)):
    """Two scans of a street, the second taken after the car turned and moved by a known step."""
    rng = np.random.default_rng(seed)
    walls = [((-40, 9), (40, 9)), ((-40, -8), (40, -8)), ((-25, -8), (-25, 9)), ((31, -8), (31, 9)), ((10, 9), (16, 6))]
    poles = rng.uniform([-35, -7], [35, 8], (12, 2))

    def look(per_m=6):
        """One scan: points at random spots on every surface, with 2 cm of noise."""
        pts = []
        for a, b in walls:
            a, b = np.array(a, float), np.array(b, float)
            u = rng.random(int(per_m * np.hypot(*(b - a))))
            pts.append(a + u[:, None] * (b - a))
        ang = rng.uniform(0, 2 * np.pi, (len(poles), 10))
        pts.append((poles[:, None, :] + 0.3 * np.stack([np.cos(ang), np.sin(ang)], -1)).reshape(-1, 2))
        p = np.vstack(pts)
        return p + rng.normal(0, 0.02, p.shape)

    a = np.radians(turn_deg)
    R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    dst = look()                                   # from the first pose
    src = (look() - np.array(shift)) @ R           # from the second pose, in its own frame
    return src, dst, {'turn_deg': turn_deg, 'shift': np.array(shift)}


src, dst, truth = test_scan_pair()
print(f"true answer: turn {truth['turn_deg']:.3f} degrees, shift {truth['shift']}")
try:
    for label, guess in (('good guess', (np.radians(3.5), (2.6, 0.0))), ('no guess  ', (0.0, (0.0, 0.0)))):
        yaw, t, rms, share = icp_2d(src, dst, *guess)
        print(f'{label}: turn {np.degrees(yaw):.3f} degrees, shift {np.round(t, 3)}, RMS {rms:.3f} m, share {share:.2f}')
except NotImplementedError as err:
    print(err)

## 5.8 An RTK outage (Figure 13)

In a tunnel, under a long bridge, or among tall buildings, GNSS can vanish for minutes. Simulate that. Pick a 90 s stretch of `car_drive` where the fix quality was 4 (RTK fixed) the whole time, with at least two turns, after the car started moving. Pretend RTK is lost at its start, and estimate the path through it three ways, each starting from the RTK position at `base_link` and the RTK heading at the start of the stretch:

1. **IMU only:** your forward velocity from 5.3 and your complementary filter heading from 5.2, integrated as in 5.5
2. **LiDAR only:** your LiDAR odometry from 5.7, restarted at the start of the stretch
3. **LiDAR with gyro heading:** the length of each LiDAR step, $|t|$, laid along the heading from the integrated gyro, with its bias removed, instead of the LiDAR's own turns

Then compare each with the RTK track you pretended to lose: the distance at every fix, and in particular after 30, 60, and 90 s.

| Figure | Content |
|---|---|
| 13 | The distance from the RTK track against time since the outage began, for the three estimates on one plot, with a log scale on the distance axis |

## Results table

Report these numbers in a table in `Lab4/README.md`. The worked example shows the format.

| Quantity | Units |
|---|---|
| Hard iron from `walk_circle`: the center of the ellipse, x and y | µT |
| Soft iron from `walk_circle`: the ratio of the ellipse's axes, and the tilt of its long axis | none, degrees |
| Hard iron from `car_circles` | µT |
| Soft iron from `car_circles` | none, degrees |
| Calibrated radius from `car_circles` | µT |
| Complementary filter cutoff $f_c$ | Hz |
| Accelerometer x bias, from the first 20 s of `car_drive` | m/s<sup>2</sup> |
| $x_c$, measured with a tape and estimated in 5.4 | m |
| $x_a$ and $y_a$, the antenna's position from `base_link`, measured with a tape | m |
| Alignment rotation from 5.5, and the declination you looked up | degrees |
| Scale factor from 5.5 | none |
| Time within 2 m and within 10 m of the RTK track | s |
| Distance from the RTK track at the end of the loop | m |
| Share of `car_drive` with RTK fixed, RTK float, and standalone | percent |
| Clock lag $L$ from `car_circles` and from `car_drive`, and the correlation at each peak | s, none |
| Clock drift between your laptop and the car PC | ppm |
| $L$ from GPS time | s |
| LiDAR odometry: time within 2 m and within 10 m of the RTK track | s |
| LiDAR odometry: distance from the RTK track at the end of the loop, and as a share of the distance driven | m, percent |
| Outage test: distance from the RTK track after 90 s, for IMU only, LiDAR only, and LiDAR with gyro heading | m |

# Analysis questions

Answer in `Lab4/README.md`, with numbers from your own data. Each answer should be a short paragraph, not an essay.

## The dry run

**Question A1. What the dry run caught.** Give the `check_bag` output for both walking bags. How long did the receiver take to reach RTK fixed, and what share of each walk was fixed? What failed or surprised you, and what did you change before the drive? What did you decide about mounting, cables, and roles for the car?

**Question A2. Calibration on foot.** Report the hard iron, the axis ratio, and the calibrated radius from `walk_circle`, and compare the radius with the horizontal field NOAA gives for your location. What on you or near you caused the hard iron? How well did your square close with each heading, and by how many degrees was your track rotated from the RTK track?

## The drive

**Question 1. Calibration in the car.** How did you calibrate the magnetometer from `car_circles`? What sources of distortion does the Mach-E have, and how can you tell from Figure 4? Compare the car calibration with the walking one.

**Question 2. The complementary filter.** Which cutoff did you choose, and why? What goes wrong with a cutoff ten times higher, and with one ten times lower? Support your answer with numbers or an extra plot.

**Question 3. Which heading to trust.** Which of your heading estimates would you trust for navigation, and when? How does your complementary filter compare with the VN-100's own yaw, and what explains the difference? Think about what the VN-100 knows about the car's iron.

**Question 4. Velocity.** Which adjustments did you make, and in what order? Give the RMS difference from the RTK speed after each one. How large was the hill term on your route? Where do your accelerometer speed and the RTK speed still disagree most, and why? How does the car's own speed compare with both?

**Question 5. The motion model.** How well does $\ddot y_{obs}$ agree with $\omega V$? Give the RMS difference before and after you add $\dot\omega\, x_c$. What else makes them differ? Report your estimated and your measured $x_c$, and explain any difference.

**Question 6. How long can you dead-reckon?** Report your rotation and scale factor. For how long after the car started moving did your track stay within 2 m of the RTK track, and within 10 m? With RTK fixed, the reference is good to a few centimeters, so these times measure your dead reckoning, not the GPS. What share of your drive was RTK fixed, float, and standalone, and how did that affect the comparison? How much did moving the RTK track to `base_link` change the 2 m time? From your results, how long could a car navigate on its IMU alone, for example through a tunnel?

**Question 7. Where the magnetometer failed.** Where on your route did the calibrated magnetometer heading differ from your complementary filter by more than 5 degrees? Use your notes, the video, and a map to say what was there. Does the difference grow when the car accelerates or brakes hard? The Mach-E is electric, and large battery currents flow under its floor when it does.

## Beyond the IMU

**Question 8. Two clocks.** Report $L$ for `car_circles` and for `car_drive`, with the correlation at each peak, the drift in ppm, and $L$ from GPS time. How well do they agree, and what explains the difference? What does Figure 11(b) look like if you forget to flip the sign, and why? Why is a lag measured from the two gyros more trustworthy than one from the times you started the two recordings?

**Question 9. LiDAR odometry.** Report the time your LiDAR track stayed within 2 m and within 10 m of the RTK track, its distance at the end of the loop, and that distance as a share of the distance driven, and compare them with your IMU dead reckoning. Where along the route did ICP have the lowest pair share or the highest RMS, and what was there? Use your notes, a map, and Figure 12. During one scan the car moves up to 1 m: what does that do to a scan on a straight street, and in a turn? On your hill, how did the LiDAR's speed compare with your accelerometer's, and why?

**Question 10. The outage.** Report the three distances after 30, 60, and 90 s. Which estimate failed fastest, and why? Why does laying the LiDAR's step lengths along the gyro heading do better than the LiDAR's own turns? A car loses GNSS in a 2 km tunnel at 20 m/s: which of your three estimates would you trust, for how long, and what would you add? Compare with your answer to Question 6.

# What to submit

**On Canvas:** one PDF with Figures 1 to 13, with your repository URL on the first page.

**In your repository:**

```
EECE5554/Lab4/
├── [NortheasternUsername]_lab4.txt   written by verify_lab4.sh, Section 10 completed
├── README.md                         the results table and the answers
├── analysis/
│   └── your notebook or scripts
├── data/
│   ├── walk_circle/                  the team's four rosbags, and the two car files
│   ├── walk_square/
│   ├── car_circles/
│   ├── car_drive/
│   └── car/
│       ├── car_circles.npz          the two car files, from Step 14
│       └── car_drive.npz
└── ros2_ws/src/
    ├── gps_msgs/
    ├── gps_driver/
    ├── vn_msgs/
    ├── vn_driver/
    └── nav_bringup/
```

Do not commit `build/`, `install/`, `log/`, or videos. A 10 minute drive makes about 15 MB of rosbag on your laptop, and each car file is about 10 MB, which is fine. Never commit the car's own bags, about 30 GB each. If you copied them with `copy team3 --raw`, keep them on your drive, outside your repository.

# Assessment (120 points)

The software and the data sets are worth 25 points, which come from your self-check report, checked against what you committed. The figures, the results table, and the questions are worth 95. Partial credit is available within each item below, and the Canvas rubric shows the exact ratings.

**Software and data sets (25 pts), from the self-check report**
<ul>
    <li> Packages and build: nav_bringup with nav.launch.py, the whole workspace builds, and both message types are there (3 pts) </li>
    <li> One launch file, two drivers: both topics publish from their own emulator in round 1 (2 pts) and in round 2 (2 pts), and the ports came from your launch arguments (2 pts) </li>
    <li> walk_circle (3 pts) and walk_square (3 pts) </li>
    <li> car_circles (3 pts) and car_drive (3 pts) </li>
    <li> The car files: car_circles.npz (1 pt) and car_drive.npz (2 pts) </li>
    <li> No build, install, or log directories committed (1 pt) </li>
</ul>

**Figures (49 pts)**
<ul>
    <li> Dry run: Figures 1, 2, and 3 (3 pts each) </li>
    <li> Drive: Figure 4 (4 pts), Figure 5 (2 pts), Figure 6 (2 pts), Figure 7 (5 pts), Figure 8 (5 pts), Figure 9 (3 pts), and Figure 10 (4 pts) </li>
    <li> Beyond the IMU: Figure 11 (4 pts), Figure 12 (6 pts), and Figure 13 (5 pts) </li>
</ul>

**Results table (5 pts)**

**Analysis questions (41 pts)**
<ul>
    <li> Questions A1 and A2 (2 pts each) </li>
    <li> Questions 1 to 7 (3 pts each) </li>
    <li> Question 8 (5 pts), Question 9 (6 pts), and Question 10 (5 pts) </li>
</ul>

### If something is not working

Push the self-check report exactly as generated, failures included, and use Section 10 to paste the exact error and your plan. An honest, specific failure report earns substantial partial credit. A hand-edited report claiming results the log does not show earns zero and is treated as an academic integrity matter. The grader cross-checks the report against what is actually committed, and may rerun the self-check on your workspace.

If your drive fails for reasons outside your control, use the course data set from the instructor for the analysis, and say so in your README.

# Appendix A: Common problems

**`/gps` publishes nothing, but the GPS emulator is streaming**
The ports are probably swapped, or `gps_port` never reaches the GPS driver. Check that your include passes `port`, not `gps_port`, to `rtk_driver.launch.py`, and read the launch terminal for the port each driver opened. On the real receiver, also check that you used its GGA port: it shows up as more than one `ttyACM` device.

**`/imu` stays at 10 Hz**
Your VN-100 driver did not write register 7, and the Lab 3 notes on that apply. If it worked in Lab 3, check that the IMU driver really opened the IMU's port.

**`package 'gps_driver' not found`**
`get_package_share_directory` cannot find a package: this terminal has not sourced your workspace, or that package did not build. `colcon build` prints a summary of what built.

**`ros2 launch` cannot find `nav.launch.py`**
`setup.py` is missing the `data_files` entry that installs `launch/`. Rebuild after fixing it.

**`/dev/serial/by-id/` does not exist**
No USB serial device is connected or, on a virtual machine, the devices are not passed through to the guest.

**A driver dies a few seconds after it starts**
Read the launch terminal for its traceback. `Device or resource busy` means another program, often minicom, still has the port open. `Permission denied` means you are not in the `dialout` group.

**The bag has only one topic**
That driver was not publishing yet when you started recording, or you listed only one topic. Run `ros2 topic hz` on both topics before you record, and `ros2 bag info` after.

**`check_bag` says the GPS fix was recorded several seconds late**
Your laptop's clock is off. Run `timedatectl` and look for `System clock synchronized: yes`. On a virtual machine, turn on time synchronization with the host. In the car, your laptop is offline on purpose, so its clock is not synchronized there: a change of a few hundredths of a second during the trip is expected, and Part 5.6 measures it.

**No GNSS fix in the car**
You are still in the garage, or the antenna is inside the car. It belongs on the roof.

**`fix_quality` stays at 1**
Corrections are not arriving. Check, in this order, as in Lab 2: the antenna sees the sky (Tracked Sats above zero on the receiver's web page), the phone hotspot is on and has signal, the CAT-6 loop is in place, and the Corrections tab shows the RTK2go stream.

**`fix_quality` drops to 5 or 1 during the drive**
That is normal near tall buildings, trees, and bridges. Keep driving, note where it happened, and use the fix quality in Part 5.

**The calibrated magnetometer is still not a circle**
The data did not turn a full 360 degrees, the IMU moved during the circles, or something magnetic moved near it, such as a phone.

**Your heading jumps by 360 degrees**
Unwrap before you filter or integrate.

**Your velocity grows without bound**
The bias or the hill term is still in it, or the velocity is never reset at stops.

**Your dead-reckoned track is a mirror image of GPS**
With heading measured clockwise from north, east is $V\sin\psi$ and north is $V\cos\psi$. Check that you did not swap them, and check the sign of your heading.

**A bag has no `metadata.yaml` after a crash**
Rebuild it with `ros2 bag reindex data/car_drive`.

**The notebook cannot import `rosbag2_py`**
Start VS Code from a terminal where ROS 2 and your workspace are sourced, and use the system Python 3.12 as the kernel.

**`check_car` notes that a message type is not installed, and the vehicle topics are missing**
The car file was made on a computer without the car's vehicle messages. Ask staff to make it again on the car PC.

**`find_lag` peaks at a correlation below 0.9**
One gyro's sign is not flipped, one curve is not gyro z, or `max_lag` is smaller than the time between the starts of the two recordings.

**Your LiDAR track turns the wrong way, or is a mirror image**
The scans use y to the left and counterclockwise angles; the VN-100 uses y to the right. Check your conversion to east and north in 5.7.

**ICP jumps, or the pair share drops**
The guess was off, the scene had too little structure, such as an open park or a wide intersection, or a bus filled the view. Check the guess from the gyro and the speed, and for a suspect step, use the guess instead of the ICP answer.

If you are stuck for more than 30 minutes, post on `Piazza` with the command you ran and the full output. Paste text, not a screenshot of a terminal.

# Academic integrity

Teams share the RTK receiver, the VN-100, the car, and all four data sets. That is expected and permitted. Everything else is individual work: your workspace and launch file, your analysis code, your figures, and your answers. Identical figures or answers across submissions will be treated as a violation.